# RARE26 — Colab training

This notebook runs the repository's configured training pipeline on a Colab GPU.
It checks out an exact Git commit, prepares the private dataset manifest, validates
every requested seed and fold, checks GPU memory at the configured batch size,
and syncs checkpoints to Drive.

Before running, upload `data.zip` and `weights.zip` to Drive, set
`EXPECTED_COMMIT_SHA`, and list the intended files in `CONFIGS_TO_RUN`. The
notebook stops before mounting Drive when either setting is missing.

## 1. Verify the GPU

In [ ]:
!nvidia-smi

## 2. Configuration — edit this cell if your Drive paths differ

Everything below assumes you uploaded `data.zip` / `weights.zip` to
`MyDrive/rare26/`. If you put them somewhere else, this is the only cell
you need to change.

In [ ]:
DRIVE_DATA_ZIP = "/content/drive/MyDrive/rare26/data.zip"
DRIVE_WEIGHTS_ZIP = "/content/drive/MyDrive/rare26/weights.zip"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/rare26/results_from_colab"

GITHUB_REPO = "mehmetaytugyuruk/rare26"
LOCAL_REPO_DIR = "/content/rare26"

# Set this to the exact public commit that should be trained.
EXPECTED_COMMIT_SHA = ""

# Select one or more repository configs explicitly.
CONFIGS_TO_RUN = []

# None runs every seed declared by each config. A positive integer limits seeds
# for a diagnostic run.
LIMIT_SEEDS = None

if (len(EXPECTED_COMMIT_SHA) != 40
        or any(ch not in "0123456789abcdef" for ch in EXPECTED_COMMIT_SHA)):
    raise ValueError("Set EXPECTED_COMMIT_SHA to the full 40-character audited commit SHA.")
if not CONFIGS_TO_RUN:
    raise ValueError("Set CONFIGS_TO_RUN explicitly.")
if any(not isinstance(path, str) or not path for path in CONFIGS_TO_RUN):
    raise ValueError("Every CONFIGS_TO_RUN entry must be a non-empty path string.")
if len(set(CONFIGS_TO_RUN)) != len(CONFIGS_TO_RUN):
    raise ValueError(f"CONFIGS_TO_RUN contains duplicates: {CONFIGS_TO_RUN}")
if (LIMIT_SEEDS is not None
        and (isinstance(LIMIT_SEEDS, bool) or not isinstance(LIMIT_SEEDS, int) or LIMIT_SEEDS <= 0)):
    raise ValueError("LIMIT_SEEDS must be a positive integer or None.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Clone the public repository

The notebook checks out `EXPECTED_COMMIT_SHA` in detached mode and verifies
the resolved commit before using any configuration.

In [ ]:
import os, shutil, subprocess

clean_url = f"https://github.com/{GITHUB_REPO}.git"
if os.path.exists(LOCAL_REPO_DIR):
    shutil.rmtree(LOCAL_REPO_DIR)

subprocess.run(["git", "clone", clean_url, LOCAL_REPO_DIR], check=True)
%cd {LOCAL_REPO_DIR}
origin_url = subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip()
assert origin_url == clean_url, f"Unexpected origin URL after clone: {origin_url}"

subprocess.run(["git", "checkout", "--detach", EXPECTED_COMMIT_SHA], check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_sha == EXPECTED_COMMIT_SHA, (
    f"SHA MISMATCH: checked out {actual_sha}, expected {EXPECTED_COMMIT_SHA}. "
    "Stop before training."
)
repo_root = os.path.realpath(LOCAL_REPO_DIR)
for cfg_path in CONFIGS_TO_RUN:
    if os.path.isabs(cfg_path):
        raise ValueError(f"CONFIGS_TO_RUN must use repo-relative paths: {cfg_path}")
    normalized_cfg = os.path.normpath(cfg_path)
    if normalized_cfg == ".." or normalized_cfg.startswith(f"..{os.sep}"):
        raise ValueError(f"Config path escapes the audited checkout: {cfg_path}")
    resolved_cfg = os.path.realpath(normalized_cfg)
    if os.path.commonpath([repo_root, resolved_cfg]) != repo_root:
        raise ValueError(f"Config resolves outside the audited checkout: {cfg_path}")
    if not os.path.isfile(normalized_cfg):
        raise FileNotFoundError(
            f"Configured experiment file is absent at audited SHA {actual_sha}: {cfg_path}"
        )
    subprocess.run(
        ["git", "ls-files", "--error-unmatch", "--", normalized_cfg],
        check=True,
        stdout=subprocess.DEVNULL,
    )
print(f"Verified: HEAD == {actual_sha}")


## 5. Copy + unzip data and weights to local Colab disk

Copying to `/content` first (local SSD) rather than reading through the
Drive FUSE mount directly — much faster for thousands of small image reads
during training.

In [ ]:
os.makedirs("/content/staging", exist_ok=True)
shutil.copy(DRIVE_DATA_ZIP, "/content/staging/data.zip")
shutil.copy(DRIVE_WEIGHTS_ZIP, "/content/staging/weights.zip")

# The private manifest is generated from the extracted dataset in a later cell.
!unzip -q -o /content/staging/data.zip -d {LOCAL_REPO_DIR}/ -x "data/data_manifest.csv"
!unzip -q -o /content/staging/weights.zip -d {LOCAL_REPO_DIR}/

# The archives should add only ignored data/weights. Abort if either one
# changed code or configs after the exact-SHA checkout was verified.
tracked_changes = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    text=True,
).strip()
assert not tracked_changes, (
    f"Drive archives modified files tracked by the audited commit:\n{tracked_changes}"
)

import glob
n_images = len(glob.glob("data/*/*/*.png"))
print(f"Images found on disk: {n_images}")
assert n_images > 0, "No PNG images found in the expected data layout."


## 6. Install the few dependencies Colab doesn't already have

Uses Colab's CUDA-matched `torch` and `torchvision` installation.

In [ ]:
!pip install -q timm pyyaml scikit-learn scipy

## 7. Sanity check the environment

In [ ]:
import sys, torch, timm
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("timm", timm.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from src.utils import load_config, get_device, get_device_config
from src.models import create_model
from src.losses import create_loss
from train import preflight_experiment
print("src/ imports OK")

subprocess.run([sys.executable, "-m", "scripts.00_prepare_manifest"], check=True)

device = get_device()
device_config = get_device_config(device)
print("Device:", device, "| device_config:", device_config)
assert device.type == "cuda", "GPU not detected -- check Runtime > Change runtime type > GPU"

# Resolve every requested seed and fold and load every requested backbone.
for cfg_path in CONFIGS_TO_RUN:
    preflight_experiment(cfg_path, limit_seeds=LIMIT_SEEDS)

print("CPU count on this instance:", os.cpu_count())

## 8. Check each configured batch size on this GPU

Runs two *real* training steps (forward, backward, optimizer step) at each
config's actual batch size, so it accounts for optimizer state rather than
just the model's forward/backward memory.

The next cell treats this as a gate and leaves every config unchanged. A
batch size that does not fit requires a deliberate experiment-design change;
the notebook must not substitute a fallback silently.

In [ ]:
import gc

def test_batch_size(config_path: str, batch_size: int) -> bool:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = optimizer = loss_fn = x = y = None
    try:
        cfg = load_config(config_path)
        model = create_model(cfg).to(device)
        if device_config["channels_last"]:
            model = model.to(memory_format=torch.channels_last)
        loss_fn = create_loss(cfg).to(device)
        lr = float(cfg["training"]["lr"])
        wd = float(cfg["training"].get("weight_decay", 1e-4))
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        scaler = torch.cuda.amp.GradScaler(enabled=device_config["use_amp"])

        img_size = cfg["data"]["img_size"]
        x = torch.randn(batch_size, 3, img_size, img_size, device=device)
        if device_config["channels_last"]:
            x = x.to(memory_format=torch.channels_last)
        y = torch.randint(0, 2, (batch_size, 1), device=device).float()

        for _ in range(2):
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device_config["use_amp"]):
                out = model(x)
                loss = loss_fn(out, y)
                # Keep the probe valid if a configured loss returns per-row values.
                loss = loss.mean()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        torch.cuda.synchronize()
        peak_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"{config_path}: batch_size={batch_size} OK, peak allocated ~{peak_gb:.2f} GB")
        return True
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"{config_path}: batch_size={batch_size} OOM")
            return False
        raise
    finally:
        del model, optimizer, loss_fn, x, y
        gc.collect()
        torch.cuda.empty_cache()

batch_fit_by_config = {}
for cfg_path in CONFIGS_TO_RUN:
    design_batch_size = load_config(cfg_path)["training"]["batch_size"]
    batch_fit_by_config[cfg_path] = {
        "batch_size": design_batch_size,
        "fits": test_batch_size(cfg_path, design_batch_size),
    }

## 9. Confirm the configured batch size fits

Each config is checked at its own `training.batch_size`. The check does not
modify configuration files. A failure stops the run.

In [ ]:
for cfg_path, result in batch_fit_by_config.items():
    design = result["batch_size"]
    assert result["fits"], (
        f"{cfg_path} wants batch_size={design}, which did NOT fit on this GPU. "
        f"Stop here -- changing batch size changes the experiment and needs "
        f"a deliberate decision, not "
        f"a silent fallback."
    )
    print(f"{cfg_path}: batch_size={design} fits, using it as-is (config untouched)")

## 10. Run the experiments

Each selected config runs through `train.py`. Its checkpoints and metadata
sidecars are synced to Drive immediately after the config completes.

In [ ]:
import sys

def sync_to_drive(cfg_path):
    """Copies this config's checkpoints to Drive. Safe to call repeatedly:
    copytree(dirs_exist_ok=True) re-syncs. The config is not copied alongside
    them -- each checkpoint's .json sidecar already carries the full merged
    config, which is strictly more than the config file's own delta."""
    exp_name = load_config(cfg_path)["experiment_name"]
    src_dir = f"models/{exp_name}"
    if not os.path.exists(src_dir):
        print(f"  {src_dir} not found, nothing to sync yet")
        return
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    shutil.copytree(src_dir, f"{DRIVE_RESULTS_DIR}/{exp_name}", dirs_exist_ok=True)
    print(f"  Synced {exp_name} to Drive")

for cfg_path in CONFIGS_TO_RUN:
    print(f"\n{'='*70}\nRunning {cfg_path}\n{'='*70}")
    command = [sys.executable, "train.py", "--config", cfg_path]
    if LIMIT_SEEDS is not None:
        command.extend(["--limit-seeds", str(LIMIT_SEEDS)])
    subprocess.run(command, check=True)
    sync_to_drive(cfg_path)

## 11. Read the results

Each cross-validation run prints pooled FPR@90, AUROC, AUPRC, official
PPV@90Recall, and centre-normalized diagnostics. Full-data runs report training-row
diagnostics and do not provide a held-out estimate. Each checkpoint's JSON
sidecar records the merged config, epoch, selection mode, and validation metrics.

## 12. Final re-sync (safety net)

Repeat the sync once after all selected configurations have completed.


In [ ]:
for cfg_path in CONFIGS_TO_RUN:
    sync_to_drive(cfg_path)

print(f"\nDone. Everything is in Drive at: {DRIVE_RESULTS_DIR}")